<a href="https://colab.research.google.com/github/SyedSameena18/CodeAlpha_SentimentAnalysis/blob/main/CODEALPHA_TASK4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ***AIRLINE PASSENGER REVIEW SENTIMENT ANALYSIS USING MACHINE LEARNING***
## Project Objective

The objective of this project is to analyze airline passenger reviews using Natural Language Processing (NLP) and Machine Learning techniques. The model classifies customer reviews into positive and negative sentiments, helping airlines understand customer satisfaction and improve service quality.



IMPORTNG LIBRARIES -

In [ ]:
# Data Manipulation
import pandas as pd
import numpy as np

# Data Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Natural Language Processing
import re
import nltk

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

# Word Cloud
from wordcloud import WordCloud

# required NLTK resources
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

LOADING DATASET -

In [ ]:
df = pd.read_csv('/content/Airline_review.csv.zip')

DATASET OVERVIEW -

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.columns.tolist()

DATA CLEANING -


In [ ]:
#missing values
df.isnull().sum()

In [ ]:
#duplicates
df.duplicated().sum()

In [ ]:
df = df[['Review', 'Recommended']]
df.head()

TARGET LABELS-

In [ ]:
df['Recommended'] = df['Recommended'].map({
    'yes': 1,
    'no': 0
})
df.head()

CLASS DISTRIBUTION-

In [ ]:
df['Recommended'].value_counts()

In [ ]:
df['Recommended'].value_counts()

We need Review (text) and Recommended (label) to build the sentiment classifier.

Mapping:

yes → 1 (Positive)

no → 0 (Negative)

# ***NLP TEXT PROCESSING-***

In [ ]:
# Text Cleaning Function

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):

    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)

    # Remove punctuation and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # Tokenize
    words = nltk.word_tokenize(text)

    # Remove stopwords and lemmatize
    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words
    ]

    return " ".join(words)

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

In [ ]:
df['Clean_Review'] = df['Review'].apply(clean_text)

In [ ]:
df[['Review', 'Clean_Review']].head()

In [ ]:
(df['Clean_Review'].str.len() == 0).sum()

TF-IDF FEATURE EXTRACTION -

In [ ]:
#Features (X) and Target (y)
X = df['Clean_Review']
y = df['Recommended']

In [ ]:
#Convert Text to TF-IDF Features
tfidf = TfidfVectorizer(max_features=5000)
X = tfidf.fit_transform(X)

In [ ]:
#Feature Matrix Shape
print(X.shape)

TRAIN-TEST SPLIT

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
#Verify the Split
print("Training Data :", X_train.shape)
print("Testing Data  :", X_test.shape)

In [ ]:
X.shape


In [ ]:
X_train.shape


In [ ]:
X_test.shape

TF-IDF converts words into numerical features while giving more importance to informative words.

Train-test split keeps 80% of the data for training and 20% for testing.

stratify=y preserves the original class distribution (about 66% negative and 34% positive) in both sets, which is important because your dataset is imbalanced.

# ***MODEL TRAINING - LOGISTIC REGRESSION MODEL***

We chose Logistic Regression because:

It is specifically designed for binary classification problems.
Our target variable (Recommended) has only two classes:

1 → Recommended (Positive)

0 → Not Recommended (Negative)

Since the goal is to predict one of two possible outcomes, Logistic Regression is a natural choice.

In [ ]:
# Train Logistic Regression Model

model = LogisticRegression(random_state=42)
model.fit(X_train, y_train)

In [ ]:
# predictions
y_pred = model.predict(X_test)

MODEL EVALUATION -

In [ ]:
#Accuracy
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

In [ ]:
#Classification report
print(classification_report(y_test, y_pred))

In [ ]:
#Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print(cm)

#VISUALIZATION
plt.figure(figsize=(6,5))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Negative','Positive'],
    yticklabels=['Negative','Positive']
)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.tight_layout()

plt.show()


The confusion matrix demonstrates the performance of the Logistic Regression model in classifying airline passenger reviews.

- 2873 negative reviews were correctly classified as negative.
- 1292 positive reviews were correctly classified as positive.
- 200 negative reviews were incorrectly classified as positive.
- 270 positive reviews were incorrectly classified as negative.

The model achieved an overall accuracy of approximately **89.86%**, indicating strong predictive performance for airline review sentiment classification.

# ***MODEL PERFORMANCE-***


- Logistic Regression was used as the classification model.
- The confusion matrix illustrates the number of correctly and incorrectly classified reviews.
- Accuracy measures the overall performance of the model on unseen data.

True Negatives (TN) = 2873
These reviews were actually negative, and the model correctly predicted them as negative.

False Positives (FP) = 200
These reviews were actually negative, but the model incorrectly predicted them as positive.

False Negatives (FN) = 270
These reviews were actually positive, but the model incorrectly predicted them as negative.

True Positives (TP) = 1292
These reviews were actually positive, and the model correctly predicted them as positive.

# ***EVALUATION METRICS -***

Accuracy: 89.86%

Precision (Positive class): 0.87

Recall (Positive class): 0.83

F1-score (Positive class): 0.85

# ***VISUALIZATION-***

In [ ]:
#1.Sentiment Distribution:
plt.figure(figsize=(6,5))
sns.countplot(
    x='Recommended',
    data=df,
    palette='Set2'
)

plt.title("Sentiment Distribution of Airline Reviews", fontsize=14)
plt.xlabel("Sentiment")
plt.ylabel("Number of Reviews")
plt.xticks(
    ticks=[0,1],
    labels=['Negative','Positive']
)
plt.tight_layout()

plt.show()


- The chart displays the distribution of positive and negative airline reviews.
- Negative reviews are more frequent than positive reviews.
- This indicates that the dataset is imbalanced, with a higher proportion of dissatisfied passengers.

In [ ]:
#2.Custom Review Prediction:
def predict_sentiment(review):
    # Clean the review
    cleaned_review = clean_text(review)
    # Convert to TF-IDF
    vector = tfidf.transform([cleaned_review])
    # Predict
    prediction = model.predict(vector)[0]
    if prediction == 1:
        return "😊 Positive Review (Recommended)"
    else:
        return "😞 Negative Review (Not Recommended)"

In [ ]:
#test1
review = "The flight was excellent. The cabin crew was friendly and the seats were comfortable."
print(predict_sentiment(review))

In [ ]:
#test2
review = "Worst airline ever. Flight was delayed for six hours and customer service was terrible."
print(predict_sentiment(review))

In [ ]:
#test3
review = "The food was average but the staff were helpful and the flight was comfortable."
print(predict_sentiment(review))


The trained Logistic Regression model successfully predicts the sentiment of new airline reviews.

By preprocessing the input text, converting it into TF-IDF features, and using the trained classifier, the model determines whether a review is :

positive (recommended) or negative (not recommended).

In [ ]:
#3.Positive Reviews Word Cloud:
positive_reviews = " ".join(
    df[df['Recommended'] == 1]['Clean_Review']
)
# Generate Word Cloud
wordcloud = WordCloud(
    width=800,
    height=400,
    background_color='white'
).generate(positive_reviews)
# Display
plt.figure(figsize=(12,6))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title("Most Frequent Words in Positive Reviews", fontsize=16)



- The word cloud highlights the most frequently occurring words in positive airline reviews.
- Larger words represent terms that appear more often.
- It provides a quick visual summary of the aspects customers appreciate most.

In [ ]:
#4.Negative Reviews Word Cloud:
negative_reviews = " ".join(
    df[df['Recommended'] == 0]['Clean_Review']
)
# Generate Word Cloud
wordcloud = WordCloud(
    width=800,
    height=400,
    background_color='white'
).generate(negative_reviews)
# Display
plt.figure(figsize=(12,6))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title("Most Frequent Words in Negative Reviews", fontsize=16)



- The word cloud highlights the most frequently occurring words in negative airline reviews.
- Larger words represent the issues mentioned most often by passengers.
- It helps identify common customer complaints and areas where airlines may need improvement.